In [ ]:
# Install required dependencies for VS Code Jupyter Notebook environment
%pip install pandas numpy scikit-learn matplotlib seaborn

# Step 1: Project Overview & Aim

## Executive Summary
This notebook implements an end-to-end Machine Learning experiment using a **Decision Tree Classifier** (`DecisionTreeClassifier`) to predict individual **Loan Approval Status** (`Loan Status`: `0 = Rejected`, `1 = Approved`).

### Business Context & Objective
Automating loan approval decisions enables financial institutions to minimize credit default risks while accelerating customer application processing. The objective of this experiment is to:
1. **Clean Explicit Anomalies**: Fix column typos (`Home Onwership` -> `Home Ownership`) and filter data entry outliers in `Age` (>100) and `Employee Experience` (>60).
2. **Handle Imbalanced Target Class**: Account for class distribution (~77.8% rejected vs ~22.2% approved) using stratified sampling (`stratify=y`).
3. **Feature Encoding**: Convert categorical variables using One-Hot Encoding (`pd.get_dummies(drop_first=True)`).
4. **Train & Interpret Model**: Train an interpretable Gini Decision Tree (`max_depth=4`, `random_state=42`) to uncover explicit decision rules.
5. **Rigorous Evaluation**: Evaluate performance using Accuracy, Precision, Recall, F1-Score, and inline Confusion Matrix heatmaps.

In [ ]:
# Standard Data Manipulation & Numerical Libraries
import pandas as pd
import numpy as np

# Visualization Libraries
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning & Evaluation Libraries (Scikit-Learn)
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

# Display configurations for seamless plotting in VS Code Notebook
%matplotlib inline
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['figure.dpi'] = 100
sns.set_theme(style="whitegrid", palette="muted")
print("✅ Libraries successfully imported and plotting configurations set.")

# Step 2: Data Loading & Initial Exploration

In this section, we load `loan_data_new.csv` and perform exploratory data analysis to inspect the raw structure, dataset dimensions, data types, and target class distribution.

In [ ]:
# Load dataset
data_path = 'loan_data_new.csv'
df_raw = pd.read_csv(data_path)

print(f"📊 Dataset Shape: {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns\n")

print("--- Data Information ---")
df_raw.info()

print("\n--- First 5 Rows ---")
display(df_raw.head())

print("\n--- Summary Statistics (Numerical Features) ---")
display(df_raw.describe().T)

print("\n--- Target Class Distribution ('Loan Status') ---")
class_counts = df_raw['Loan Status'].value_counts()
class_props = df_raw['Loan Status'].value_counts(normalize=True) * 100
class_dist = pd.DataFrame({'Count': class_counts, 'Percentage (%)': class_props})
display(class_dist)

# Step 3: Data Cleaning & Preprocessing

### Explicit Dataset Anomalies Addressed:
1. **Column Name Typo Fix**: Renamed `Home Onwership` -> `Home Ownership`.
2. **Outlier Filtering**:
   - `Age`: Filtered rows where `Age > 100` (contains invalid values up to 144).
   - `Employee Experience`: Filtered rows where `Employee Experience > 60` (contains invalid values up to 125).
3. **Categorical Encoding**:
   - Applied One-Hot Encoding (`pd.get_dummies(drop_first=True)`) to categorical variables: `Gender`, `Education`, `Home Ownership`, `Loan Intent`, and `Previous Loan`.

In [ ]:
# 1. Fix Column Name Typo
df_cleaned = df_raw.rename(columns={'Home Onwership': 'Home Ownership'}).copy()
print("1. Fixed Column Name Typo: 'Home Onwership' -> 'Home Ownership'")

# 2. Outlier and Data Entry Error Filtering
initial_rows = len(df_cleaned)

# Remove invalid Age entries (> 100)
age_outliers = (df_cleaned['Age'] > 100).sum()
df_cleaned = df_cleaned[df_cleaned['Age'] <= 100]

# Remove invalid Employee Experience entries (> 60)
exp_outliers = (df_cleaned['Employee Experience'] > 60).sum()
df_cleaned = df_cleaned[df_cleaned['Employee Experience'] <= 60]

final_rows = len(df_cleaned)
removed_rows = initial_rows - final_rows

print(f"2. Outlier Removal Summary:")
print(f"   - Age > 100 rows removed: {age_outliers}")
print(f"   - Employee Experience > 60 rows removed: {exp_outliers}")
print(f"   - Total rows removed: {removed_rows} ({(removed_rows/initial_rows)*100:.2f}%)")
print(f"   - Remaining Clean Rows: {final_rows:,}\n")

# Verify extreme values post-cleaning
print("Verification of Max Values Post-Cleaning:")
print(f"   - Max Age: {df_cleaned['Age'].max()}")
print(f"   - Max Employee Experience: {df_cleaned['Employee Experience'].max()}\n")

# 3. Categorical One-Hot Encoding
categorical_cols = ['Gender', 'Education', 'Home Ownership', 'Loan Intent', 'Previous Loan']
print(f"3. Applying One-Hot Encoding (drop_first=True) on: {categorical_cols}")

# One-Hot Encoding
df_encoded = pd.get_dummies(df_cleaned, columns=categorical_cols, drop_first=True, dtype=int)

print(f"\nEncoded Dataset Shape: {df_encoded.shape[0]:,} rows x {df_encoded.shape[1]} columns")
print("\nFirst 5 Rows of Preprocessed & Encoded Feature Matrix:")
display(df_encoded.head())

# Step 4: Train-Test Split & Model Training

### Strategy & Methodology:
- **Feature Matrix ($X$) & Target ($y$)**: Separate input predictor columns from `Loan Status`.
- **Stratified Train-Test Split (80/20)**: Use `stratify=y` and `random_state=42` to maintain the ~77.8% / ~22.2% class distribution ratio across both subsets.
- **Decision Tree Model Configuration**: Instantiate `DecisionTreeClassifier(criterion='gini', max_depth=4, random_state=42)`. Setting `max_depth=4` ensures high model interpretability and guards against overfitting.

In [ ]:
# Separate Features (X) and Target (y)
X = df_encoded.drop(columns=['Loan Status'])
y = df_encoded['Loan Status']

# 80/20 Stratified Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, 
    y, 
    test_size=0.20, 
    stratify=y, 
    random_state=42
)

print(f"Feature Matrix (X) Shape: {X.shape}")
print(f"Training Set (X_train, y_train): {X_train.shape[0]:,} samples")
print(f"Testing Set (X_test, y_test):   {X_test.shape[0]:,} samples\n")
print(f"Train Target Class Distribution Ratio:\n{y_train.value_counts(normalize=True)}\n")
print(f"Test Target Class Distribution Ratio:\n{y_test.value_counts(normalize=True)}\n")

# Instantiate DecisionTreeClassifier
dt_clf = DecisionTreeClassifier(
    criterion='gini',
    max_depth=4,
    random_state=42
)

# Fit Model on Training Data
dt_clf.fit(X_train, y_train)
print("✅ Decision Tree Classifier trained successfully!")

# Step 5: Model Evaluation & Performance Metrics

We evaluate the Decision Tree model on unseen test data (`X_test`) using key performance metrics:
- **Accuracy**: Overall fraction of correct predictions.
- **Precision**: Proportion of true loan approvals out of all predicted approvals.
- **Recall (Sensitivity)**: Proportion of actual loan approvals correctly captured.
- **F1-Score**: Harmonic mean of Precision and Recall.
- **Confusion Matrix**: Heatmap visualization showing True Positives, True Negatives, False Positives, and False Negatives.

In [ ]:
# Predict outcomes on test dataset
y_pred = dt_clf.predict(X_test)

# Calculate evaluation metrics
acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print("=" * 52)
print("         MODEL PERFORMANCE EVALUATION METRICS       ")
print("=" * 52)
print(f"  Accuracy  : {acc * 100:.2f}%")
print(f"  Precision : {prec * 100:.2f}%")
print(f"  Recall    : {rec * 100:.2f}%")
print(f"  F1-Score  : {f1 * 100:.2f}%")
print("=" * 52)

print("\n--- Detailed Classification Report ---")
print(classification_report(y_test, y_pred, target_names=['Rejected (0)', 'Approved (1)']))

# Compute Confusion Matrix
cm = confusion_matrix(y_test, y_pred)

# Plot Seaborn Confusion Matrix Heatmap
plt.figure(figsize=(7, 5), dpi=120)
sns.heatmap(
    cm, 
    annot=True, 
    fmt='d', 
    cmap='Blues', 
    cbar=True,
    xticklabels=['Rejected (0)', 'Approved (1)'],
    yticklabels=['Rejected (0)', 'Approved (1)'],
    annot_kws={"size": 14, "weight": "bold"}
)
plt.title('Confusion Matrix - Decision Tree Model', fontsize=14, pad=12, fontweight='bold')
plt.xlabel('Predicted Label', fontsize=12, fontweight='bold')
plt.ylabel('Actual Label', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

# Step 6: Decision Tree Visualization

Plotting the full decision tree architecture using `plot_tree()` provides transparent visual inspection of node splitting thresholds, Gini impurities, sample sizes, and leaf node predictions.

In [ ]:
# Plot High-Resolution Decision Tree Diagram
plt.figure(figsize=(24, 12), dpi=300)

plot_tree(
    dt_clf,
    feature_names=X.columns.tolist(),
    class_names=['Rejected', 'Approved'],
    filled=True,
    rounded=True,
    fontsize=10,
    precision=2
)

plt.title("Decision Tree Architecture (Gini Impurity, max_depth=4)", fontsize=18, fontweight='bold', pad=20)
plt.tight_layout()
# Save figure to file
plt.savefig('decision_tree_diagram.png', dpi=300, bbox_inches='tight')
plt.show()
print("🖼️ Decision Tree diagram saved as 'decision_tree_diagram.png'.")

# Step 7: Conclusion & Key Findings

### Key Experiment Takeaways:
1. **Data Cleaning Effectiveness**: Outliers in `Age` (>100) and `Employee Experience` (>60) were cleanly removed, preventing extreme boundary distortion.
2. **Target Balance Handling**: Stratified splitting (`stratify=y`) preserved the natural ~77.8% / ~22.2% class distribution ratio, leading to stable cross-split metrics.
3. **Dominant Decision Rules**:
   - **`Loan percentage`**: Serves as the primary root splitting criterion. Applicants with a high loan percentage relative to income face significantly higher rejection likelihood.
   - **`Person Income` & `Credit Score`**: Serve as critical sub-branch decision rules separating creditworthy applicants from defaults.
   - **`Home Ownership`**: Categorical home status further refines loan approval probability.

### Future Recommendations:
- Explore Ensemble Models (Random Forest, Gradient Boosting / XGBoost) for potential gain in recall on minority positive class (Approved).
- Implement Hyperparameter Optimization (GridSearchCV / RandomizedSearchCV) tuning `min_samples_split`, `min_samples_leaf`, and cost-complexity pruning (`ccp_alpha`).